In [2]:
import pandas as pd

df = pd.read_csv('layoffs.csv')

# Basic info
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nFirst 5 rows:")
df.head()

Shape: (4435, 11)

Columns: ['company', 'location', 'total_laid_off', 'date', 'percentage_laid_off', 'industry', 'source', 'stage', 'funds_raised', 'country', 'date_added']

Data types:
 company                    str
location                   str
total_laid_off         float64
date                       str
percentage_laid_off    float64
industry                   str
source                     str
stage                      str
funds_raised           float64
country                    str
date_added                 str
dtype: object

Missing values:
 company                   0
location                  1
total_laid_off         1529
date                      0
percentage_laid_off    1643
industry                  2
source                    3
stage                     5
funds_raised            513
country                   2
date_added                0
dtype: int64

First 5 rows:


,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added
0,Google,SF Bay Area,NaN,6/4/2026,NaN,Consumer,https://www.businessinsider.com/google-clouds-...,Post-IPO,26.0,United States,6/7/2026
1,Skai,"Tel Aviv, Non-U.S.",100.0,6/3/2026,0.20,Marketing,https://www.calcalistech.com/ctechnews/article...,Series E,60.0,Israel,6/7/2026
2,Uber,SF Bay Area,NaN,6/3/2026,0.01,Transportation,https://www.cnbc.com/2026/06/03/uber-layoffs-p...,Post-IPO,25200.0,United States,6/4/2026
3,GitLab,SF Bay Area,350.0,6/2/2026,0.14,Product,https://www.wsj.com/business/earnings/gitlab-t...,Post-IPO,413.0,United States,5/11/2026
4,Aleph Farms,"Tel Aviv, Non-U.S.",10.0,6/2/2026,NaN,Food,https://www.calcalistech.com/ctechnews/article...,Unknown,119.0,Israel,6/3/2026


In [3]:
import pandas as pd

df = pd.read_csv('layoffs.csv')

# 1. Convert date columns to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# 2. Extract year and month (useful for charts later)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['year_month'] = df['date'].dt.to_period('M').astype(str)

# 3. Fill missing numerical values with 0
df['total_laid_off'] = df['total_laid_off'].fillna(0).astype(int)
df['funds_raised'] = df['funds_raised'].fillna(0)

# 4. Fill missing text values with 'Unknown'
df['industry'] = df['industry'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['location'] = df['location'].fillna('Unknown')
df['stage'] = df['stage'].fillna('Unknown')

# 5. Drop rows where percentage_laid_off is missing AND total_laid_off is 0
# (these rows have no useful data at all)
df = df[~((df['total_laid_off'] == 0) & (df['percentage_laid_off'].isna()))]

# 6. Drop the source column (not useful for our analysis)
df = df.drop(columns=['source'])

# Check the result
print("Shape after cleaning:", df.shape)
print("\nMissing values after cleaning:")
print(df.isnull().sum())
print("\nDate range:", df['date'].min(), "to", df['date'].max())
df.head()

Shape after cleaning: (3713, 13)

Missing values after cleaning:
company                  0
location                 0
total_laid_off           0
date                     0
percentage_laid_off    921
industry                 0
stage                    0
funds_raised             0
country                  0
date_added               0
year                     0
month                    0
year_month               0
dtype: int64

Date range: 2020-03-11 00:00:00 to 2026-06-03 00:00:00


,company,location,total_laid_off,date,percentage_laid_off,industry,stage,funds_raised,country,date_added,year,month,year_month
1,Skai,"Tel Aviv, Non-U.S.",100,2026-06-03,0.20,Marketing,Series E,60.0,Israel,2026-06-07,2026,6,2026-06
2,Uber,SF Bay Area,0,2026-06-03,0.01,Transportation,Post-IPO,25200.0,United States,2026-06-04,2026,6,2026-06
3,GitLab,SF Bay Area,350,2026-06-02,0.14,Product,Post-IPO,413.0,United States,2026-05-11,2026,6,2026-06
4,Aleph Farms,"Tel Aviv, Non-U.S.",10,2026-06-02,NaN,Food,Unknown,119.0,Israel,2026-06-03,2026,6,2026-06
5,Manhattan Associates,Atlanta,0,2026-06-02,0.06,Logistics,Post-IPO,0.0,United States,2026-06-03,2026,6,2026-06


In [4]:
import matplotlib.pyplot as plt

yearly = df.groupby('year')['total_laid_off'].sum().reset_index()
print(yearly)

   year  total_laid_off
0  2020           80998
1  2021           15823
2  2022          165269
3  2023          264320
4  2024          152922
5  2025          124636
6  2026          116839


In [5]:
top_companies = df.groupby('company')['total_laid_off'].sum().sort_values(ascending=False).head(15)
print(top_companies)

company
Amazon        58124
Intel         43115
Meta          35700
Oracle        31294
Microsoft     30055
Dell          23650
Cisco         18521
Salesforce    16525
Tesla         14500
Google        13697
SAP           11000
Ericsson      10100
Philips       10000
PayPal         9428
HP             8100
Name: total_laid_off, dtype: int64


In [6]:
top_industries = df.groupby('industry')['total_laid_off'].sum().sort_values(ascending=False).head(10)
print(top_industries)

industry
Other             119746
Retail            106706
Hardware          106157
Consumer           97007
Finance            67682
Transportation     66002
Food               52101
Healthcare         39244
Infrastructure     24835
Travel             23740
Name: total_laid_off, dtype: int64


In [7]:
top_countries = df.groupby('country')['total_laid_off'].sum().sort_values(ascending=False).head(10)
print(top_countries)

country
United States     656788
India              66039
Germany            32055
United Kingdom     23354
Netherlands        21575
Sweden             20379
Canada             16068
Israel             14494
Brazil             11939
China               8190
Name: total_laid_off, dtype: int64


In [8]:
monthly = df.groupby('year_month')['total_laid_off'].sum().reset_index()
monthly = monthly.sort_values('year_month')
print(monthly.tail(20))

   year_month  total_laid_off
55    2024-11            6755
56    2024-12            2568
57    2025-01            2537
58    2025-02           18474
59    2025-03            8914
60    2025-04           25420
61    2025-05           10577
62    2025-06            1606
63    2025-07           16648
64    2025-08            6452
65    2025-09            4454
66    2025-10           19070
67    2025-11            8932
68    2025-12            1552
69    2026-01           25148
70    2026-02           11019
71    2026-03           46536
72    2026-04            4712
73    2026-05           28889
74    2026-06             535
